In [1]:
!pip install transformers accelerate torch torchvision bitsandbytes
!pip install tqdm
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
import json
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import random
import datetime
import traceback
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoTokenizer,
    AutoProcessor,
    BitsAndBytesConfig
)
from difflib import SequenceMatcher
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm
from collections import Counter

class QwenVQAEvaluator:
    def __init__(self, config_path):
        with open(config_path) as f:
            self.config = json.load(f)

        self.model_config = self.config["open_source_models"]["qwen"]
        self.sampling_percentage = self.config.get("sampling_percentage", 100)
        self.unable_to_respond_aware = self.config.get("unable_to_respond_aware", True)
        self.initialize_model()


    def _create_prompt(self, question, structured_text, layout_summary):
        prompt = (
            "You are a highly precise AI assistant for document analysis.\n"
            "Your task is to answer questions about the document image content precisely.\n\n"
            "DOCUMENT CONTENT\n"
            f"{structured_text}\n\n"
            f"{layout_summary}\n\n"
            "GUIDELINES\n"
            "1. Identify Key Entities: Extract the main entities or concepts from the question.\n"
            "2. Identify and Categorize Document Elements: Examine the document content to identify the distinct elements present (e.g., [Title], [Plain Text], [Table], [Figure], ecc). Use the document structure summary above to assess the overall composition of the document.\n"
            "3. Check for Key Question Entities in Each Document Element: Look for matches between identified question entities inside document elements.\n"
            "4. Check for Consistency Between Document Elements and Question Context: Evaluate whether the element containing the match (e.g., [Title], [Plain Text], [Endnote]) aligns with the question’s intent. If information appears only in secondary elements or mismatched contexts, treat as invalid.\n"
            "5. Check for Contradictions Between Document Elements and the Question: Examine whether any document element contains information that directly contradicts the facts or assumptions stated in the question. If any element presents a clear contradiction or conflicting information, respond 'Unable to determine'.\n"
            "6. Formulate the Answer: If a valid and consistent match exists, provide a concise factual answer (single word or short phrase). If entities are missing, contexts are mismatched, or any contradiction is detected, respond 'Unable to determine'.\n\n"
            f"QUESTION\n"
            f"{question}\n"
            "Final Answer:"
        )
        return prompt 


    def initialize_model(self):
        print("Initializing Qwen model...")
        model_name = self.model_config["model_name"]

        # Configurazione per la quantizzazione a 8-bit
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_threshold=6.0,  # valore di default, puoi regolare se serve
            llm_int8_has_fp16_weight=False
        )

        self.processor = AutoProcessor.from_pretrained(model_name)

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_name,
            cache_dir="/data1/hf_cache/models",
            device_map="auto",
            quantization_config=quantization_config  # <-- Attiva la quantizzazione 8-bit
        ).eval()

        self.max_tokens = self.model_config.get("max_tokens", 1024)
        print("Qwen model initialized successfully with 8-bit quantization")

    def _cleanup_model(self):
        if hasattr(self, "model"):
            del self.model
            torch.cuda.empty_cache()
            import gc

            gc.collect()

    def get_sorted_ocr_text(self, layout_analysis):
        """Extract and sort OCR text by bounding box position"""
        ocr_items = []
        for obj in layout_analysis.values():
            if isinstance(obj, dict) and "OCR" in obj and "BBOX" in obj:
                bbox = obj["BBOX"]
                ocr_items.append((bbox[1], bbox[0], obj["OCR"]))  # y, x, text

        # Sort by y coordinate first, then x coordinate
        ocr_items.sort()
        return "\n".join(item[2] for item in ocr_items)

    def get_structured_ocr_text(self, pages_layout):
        full_structured_text = []
        for page_id, page_data in pages_layout.items():
            image_filename = os.path.basename(page_id)
            full_structured_text.append(f"--- Page: {image_filename} ---")
            layout_objects = page_data.get("layout_analysis", {})
            sorted_objects = sorted(
                layout_objects.values(),
                key=lambda obj: (obj.get("BBOX", [0, 0])[1], obj.get("BBOX", [0, 0])[0])
            )
            for obj in sorted_objects:
                obj_type = obj.get("ObjectType", "unknown").replace("_", " ").title()
                ocr_text = obj.get("OCR", "").strip()
                if ocr_text:
                    full_structured_text.append(f"[{obj_type}]: {ocr_text}")
        return "\n".join(full_structured_text)

    from collections import Counter

    def get_document_layout_summary(self, pages):
        """
        Generate a detailed summary of ObjectType counts and percentages
        for each document page.
        """
        if not pages:
            return "No layout information available."
        
        summary_lines = []
        for page_id, page_data in pages.items():
            layout_objs = page_data.get("layout_analysis", {})
            counter = Counter()
    
            for obj in layout_objs.values():
                obj_type = obj.get("ObjectType", "Unknown")
                counter[obj_type] += 1
    
            total_objects = sum(counter.values()) or 1
            formatted_items = [
                f"{obj_type}: {count} ({(count / total_objects) * 100:.1f}%)"
                for obj_type, count in sorted(counter.items())
            ]
            summary_lines.append(f"- {os.path.basename(page_id)} → {', '.join(formatted_items)}")
    
        return (
            "### DOCUMENT STRUCTURE SUMMARY ###\n"
            "Below is the distribution of visual Document element types detected in the document pages:\n"
            + "\n".join(summary_lines)
        )



    def generate_answer(self, question, image_paths, structured_text, layout_summary):
        try:
            window_size = self.model_config.get("batch_size", 1)
            stride = self.model_config.get("stride", window_size // 2) if window_size > 1 else 1
            total_images = len(image_paths)
            total_windows = max(1, (total_images - window_size + stride) // stride)
            all_responses = []

            for window_idx in range(total_windows):
                start_idx = window_idx * stride
                end_idx = min(start_idx + window_size, total_images)
                window_paths = image_paths[start_idx:end_idx]
                if window_idx == total_windows - 1 and end_idx < total_images:
                    window_paths = image_paths[-window_size:]

                question_prompt = self._create_prompt(question, structured_text, layout_summary)
                messages = [{
                    "role": "user",
                    "content": [
                        *[{"type": "image", "image": f"file://{path}"} for path in window_paths],
                        {"type": "text", "text": question_prompt},
                    ],
                }]

                text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                image_inputs, video_inputs = process_vision_info(messages)
                inputs = self.processor(
                    text=[text],
                    images=image_inputs,
                    videos=video_inputs,
                    padding=True,
                    return_tensors="pt",
                ).to("cuda")

                generated_ids = self.model.generate(**inputs, max_new_tokens=self.max_tokens)
                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]
                response = self.processor.batch_decode(
                    generated_ids_trimmed,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False,
                )[0]

                torch.cuda.empty_cache()
                import gc
                gc.collect()

                all_responses.append({
                    "pages": window_paths,
                    "answer": response,
                })

            return {
                "answer": all_responses,
                "query": question,
                "image_paths": image_paths,
                "analysis_type": f"window_size_{window_size}",
            }

        except Exception as e:
            print(f"Error in generate_answer: {str(e)}")
            print(f"Full error: {traceback.format_exc()}")
            return {
                "answer": "Unable to determine: error",
                "error": str(e),
                "traceback": traceback.format_exc(),
            }

    def _save_results(self, data):
        output_file = self.model_config["name"] + "_" + self.config["output_file"]
        if self.config["ocr_enabled"] and not self.config["unable_to_respond_aware"]:
            output_file = output_file.replace(".json", "_OCR_UNABLE.json")
        elif self.config["ocr_enabled"]:
            output_file = output_file.replace(".json", "_OCR.json")
        elif not self.config["unable_to_respond_aware"]:
            output_file = output_file.replace(".json", "_UNABLE.json")

        try:
            with open(output_file, "w") as f:
                json.dump(data, f, indent=2)
            print(f"Results successfully saved to {output_file}")
        except Exception as e:
            print(f"Error saving results: {str(e)}")

    def evaluate(self):
        try:
            print("\nStarting Qwen evaluation...")
            with open(self.config["input_file"]) as f:
                data = json.load(f)
                print(f"Successfully loaded input file: {self.config['input_file']}")

            total_questions = len(data["corrupted_questions"])
            num_samples = int(total_questions * (self.sampling_percentage / 100))

            if self.sampling_percentage < 100:
                sampled_questions = random.sample(data["corrupted_questions"], num_samples)
                data["corrupted_questions"] = sampled_questions
                print(f"Sampled {num_samples} questions ({self.sampling_percentage}%) for evaluation")
            else:
                print("Processing 100% of questions (no sampling)")

            processed_count = 0
            success_count = 0
            error_count = 0

            for item in tqdm(data["corrupted_questions"]):
                try:
                    processed_count += 1
                    item.setdefault("verification_result", {}).setdefault("vqa_results", [])
                    question = item["corrupted_question"]
                    pages = item["layout_analysis"]["pages"]
                    structured_text = self.get_structured_ocr_text(pages)
                    layout_summary = self.get_document_layout_summary(pages)
                    image_paths = [os.path.join(self.config["images_base_path"], os.path.basename(page_id)) for page_id in pages]
                    result = self.generate_answer(question, image_paths, structured_text, layout_summary)

                    vqa_result = {
                        "model_type": "qwen",
                        "model_config": {
                            "batch_size": self.model_config.get("batch_size", 1),
                            "max_tokens": self.max_tokens,
                            "use_flash_attention": self.model_config.get("use_flash_attention", False)
                        },
                        "ocr_enabled": bool(structured_text),
                        "question": question,
                        "answer": result.get("answer", "Unable to determine"),
                        "image_paths": image_paths,
                        "analysis_type": result.get("analysis_type", ""),
                        "structured_ocr": structured_text,
                        "layout_summary": layout_summary,
                        "timestamp": datetime.datetime.now().isoformat()
                    }

                    if "error" in result:
                        vqa_result["error"] = result["error"]
                        vqa_result["traceback"] = result.get("traceback", "")
                        error_count += 1
                    else:
                        success_count += 1

                    item["verification_result"]["vqa_results"].append(vqa_result)

                except Exception as e:
                    print(f"Error processing question: {str(e)}")
                    print(f"Full error: {traceback.format_exc()}")
                    error_count += 1

            print(f"\nProcessing completed:")
            print(f"Total questions processed: {processed_count}")
            print(f"Successful generations: {success_count}")
            print(f"Errors encountered: {error_count}")
            if processed_count > 0:
                print(f"Success rate: {(success_count/processed_count)*100:.2f}%")

            self._save_results(data)

        except Exception as e:
            print(f"Critical error in evaluate: {str(e)}")
            print(f"Full error: {traceback.format_exc()}")
        finally:
            self._cleanup_model()


def main():
    config_path = "/kaggle/input/kaggle-config/config_kaggle.json"
    evaluator = QwenVQAEvaluator(config_path)
    evaluator.evaluate()


if __name__ == "__main__":
    main()

2025-11-12 14:40:59.762058: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762958459.941436      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762958459.991856      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Initializing Qwen model...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen model initialized successfully with 8-bit quantization

Starting Qwen evaluation...
Successfully loaded input file: /kaggle/input/dude-questions/DUDE_fixed.json
Processing 100% of questions (no sampling)


  0%|          | 0/187 [00:00<?, ?it/s]

Error in generate_answer: CUDA out of memory. Tried to allocate 7.88 GiB. GPU 0 has a total capacity of 14.74 GiB of which 6.53 GiB is free. Process 4071 has 8.21 GiB memory in use. Of the allocated memory 7.33 GiB is allocated by PyTorch, and 763.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Full error: Traceback (most recent call last):
  File "/tmp/ipykernel_19/39294644.py", line 179, in generate_answer
    generated_ids = self.model.generate(**inputs, max_new_tokens=self.max_tokens)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/